# Clickstream Behavioral Analysis
**Sequential patterns · Markov chains · Time-to-purchase · Predictive models · CLV**

Place all `.parquet` files in the working directory before running.

**Extended version** — adds: XGBoost, LightGBM, Logistic Regression benchmarking, SHAP feature attribution, Average Precision metric, and `purchase_rate` feature.

## 0. Setup & DuckDB Configuration

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os
import warnings
import json
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

con = duckdb.connect()

# Memory: use ~80% of available RAM. Adjust to your machine.
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=4")
con.execute("SET preserve_insertion_order=false")

# Allow DuckDB to spill to disk when memory is tight.
import tempfile
tmp_dir = tempfile.gettempdir()
con.execute(f"SET temp_directory='{tmp_dir}'")
con.execute("SET max_temp_directory_size='100GB'")

print(f'DuckDB ready  |  temp dir: {tmp_dir}')

DuckDB ready  |  temp dir: C:\Users\kirez\AppData\Local\Temp


## 1. Load Events

In [2]:
con.execute("""
CREATE OR REPLACE TABLE events AS
SELECT * FROM read_parquet('*.parquet')
""")

total = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
users = con.execute("SELECT COUNT(DISTINCT user_id) FROM events").fetchone()[0]
print(f'Events loaded: {total:,}  |  Unique users: {users:,}')

print('\nSchema:')
print(con.execute('DESCRIBE events').df())

print('\nEvent type breakdown:')
event_breakdown = con.execute("""
    SELECT event_type, COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM events
    GROUP BY event_type
    ORDER BY n DESC
""").df()
print(event_breakdown)
event_breakdown.to_csv('outputs/event_breakdown.csv', index=False)

Events loaded: 269,878,620  |  Unique users: 12,124,772

Schema:
             column_name               column_type null   key default extra
0             event_time  TIMESTAMP WITH TIME ZONE  YES  None    None  None
1             event_type                   VARCHAR  YES  None    None  None
2             product_id                    BIGINT  YES  None    None  None
3            category_id                    BIGINT  YES  None    None  None
4          category_code                   VARCHAR  YES  None    None  None
5                  brand                   VARCHAR  YES  None    None  None
6                  price                    DOUBLE  YES  None    None  None
7                user_id                    BIGINT  YES  None    None  None
8           user_session                   VARCHAR  YES  None    None  None
9      brand_was_missing                   INTEGER  YES  None    None  None
10  category_was_missing                   INTEGER  YES  None    None  None
11                  hou

## 2. Session Table (Daily Proxy)

In [3]:
con.execute("""
CREATE OR REPLACE TABLE sessions AS
SELECT
    user_id,
    DATE_TRUNC('day', event_time) AS session_day,
    MIN(event_time)               AS session_start,
    MAX(event_time)               AS session_end,
    COUNT(*)                      AS event_count,
    SUM(CASE WHEN event_type = 'view'     THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchases
FROM events
GROUP BY user_id, session_day
""")

print('Sessions table created')
sess_stats = con.execute("""
    SELECT
        COUNT(*)                                                        AS total_sessions,
        ROUND(AVG(event_count), 1)                                      AS avg_events_per_session,
        ROUND(AVG(DATEDIFF('minute', session_start, session_end)), 1)   AS avg_session_minutes
    FROM sessions
""").df()
print(sess_stats)

Sessions table created
   total_sessions  avg_events_per_session  avg_session_minutes
0        38870618                     6.9                 63.7


## 3. Markov Chain — Transition Probabilities

Memory-safe approach: rank-based self-join instead of `LEAD()` over 270M rows.
Row-normalised first-order transition matrix: $P(e_{i+1} \mid e_i)$.

In [4]:
transition_probs = con.execute("""
WITH ranked AS (
    SELECT
        user_id,
        event_type,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_time) AS rn
    FROM events
),
pairs AS (
    SELECT
        a.event_type AS from_event,
        b.event_type AS to_event
    FROM ranked a
    JOIN ranked b
      ON a.user_id = b.user_id
     AND b.rn = a.rn + 1
)
SELECT
    from_event,
    to_event,
    COUNT(*)                                                                   AS count,
    ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY from_event), 4)   AS prob
FROM pairs
GROUP BY from_event, to_event
ORDER BY from_event, prob DESC
""").df()

print('--- Transition Probabilities ---')
print(transition_probs)
transition_probs.to_csv('outputs/transition_probs.csv', index=False)

transition_matrix = (
    transition_probs
    .pivot(index='from_event', columns='to_event', values='prob')
    .fillna(0)
    .round(4)
)
print('\n--- Transition Matrix ---')
print(transition_matrix)
transition_matrix.to_csv('outputs/transition_matrix.csv')

top_transitions = transition_probs.sort_values('prob', ascending=False).head(10)
print('\n--- Top 10 Transitions ---')
print(top_transitions)
top_transitions.to_csv('outputs/top_transitions.csv', index=False)

--- Transition Probabilities ---
  from_event  to_event      count    prob
0       cart      view    6110098  0.5271
1       cart  purchase    3418480  0.2949
2       cart      cart    2062639  0.1779
3   purchase      view    3766351  0.9701
4   purchase  purchase      86287  0.0222
5   purchase      cart      29718  0.0077
6       view      view  231874476  0.9571
7       view      cart    9762944  0.0403
8       view  purchase     642855  0.0027

--- Transition Matrix ---
to_event      cart  purchase    view
from_event                          
cart        0.1779    0.2949  0.5271
purchase    0.0077    0.0222  0.9701
view        0.0403    0.0027  0.9571

--- Top 10 Transitions ---
  from_event  to_event      count    prob
3   purchase      view    3766351  0.9701
6       view      view  231874476  0.9571
0       cart      view    6110098  0.5271
1       cart  purchase    3418480  0.2949
2       cart      cart    2062639  0.1779
7       view      cart    9762944  0.0403
4   purchase 

## 4. Conversion Funnel

In [5]:
funnel = con.execute("""
WITH user_summary AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_type = 'view'     THEN 1 ELSE 0 END) AS did_view,
        MAX(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) AS did_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS did_purchase
    FROM events
    GROUP BY user_id
)
SELECT
    COUNT(*)                                          AS total_users,
    SUM(did_view)                                     AS viewed,
    SUM(did_cart)                                     AS carted,
    SUM(did_purchase)                                 AS purchased,
    SUM(CASE WHEN did_cart = 1
              AND did_purchase = 0 THEN 1 ELSE 0 END) AS cart_abandoners
FROM user_summary
""").df()

print('--- Conversion Funnel ---')
print(funnel.T)

total_u   = funnel['total_users'].iloc[0]
viewed    = funnel['viewed'].iloc[0]
carted    = funnel['carted'].iloc[0]
purchased = funnel['purchased'].iloc[0]

print(f'\nView rate:        {viewed/total_u*100:.1f}%')
print(f'Cart rate:        {carted/viewed*100:.1f}%  (of viewers)')
print(f'Purchase rate:    {purchased/carted*100:.1f}%  (of carted users)')
print(f'Overall CVR:      {purchased/total_u*100:.1f}%')
print(f'Cart abandonment: {funnel["cart_abandoners"].iloc[0]/carted*100:.1f}%  (of carted)')

funnel.to_csv('outputs/funnel.csv', index=False)

--- Conversion Funnel ---
                          0
total_users      12124772.0
viewed           12121513.0
carted            2721066.0
purchased         1547532.0
cart_abandoners   1271473.0

View rate:        100.0%
Cart rate:        22.4%  (of viewers)
Purchase rate:    56.9%  (of carted users)
Overall CVR:      12.8%
Cart abandonment: 46.7%  (of carted)


## 5. Time-to-Purchase Analysis

In [6]:
time_to_purchase = con.execute("""
SELECT
    user_id,
    MIN(event_time) FILTER (WHERE event_type = 'view')     AS first_view,
    MIN(event_time) FILTER (WHERE event_type = 'purchase') AS purchase_time,
    CASE
        WHEN COUNT(*) FILTER (WHERE event_type = 'purchase') > 0 THEN 1
        ELSE 0
    END AS converted
FROM events
GROUP BY user_id
""").df()

time_to_purchase['time_to_purchase_sec'] = (
    time_to_purchase['purchase_time'] - time_to_purchase['first_view']
).dt.total_seconds()

ttp_clean = time_to_purchase[
    (time_to_purchase['converted'] == 1) &
    (time_to_purchase['time_to_purchase_sec'] >= 0)
].copy()

ttp_hours = ttp_clean['time_to_purchase_sec'] / 3600

print('--- Time-to-Purchase (converted users) ---')
print(f'Count:    {len(ttp_clean):,}')
print(f'Mean:     {ttp_hours.mean():.2f} hrs')
print(f'Median:   {ttp_hours.median():.2f} hrs')
print(f'P25:      {ttp_hours.quantile(0.25):.2f} hrs')
print(f'P75:      {ttp_hours.quantile(0.75):.2f} hrs')
print(f'P95:      {ttp_hours.quantile(0.95):.2f} hrs')

print('\nBucket distribution:')
bins   = [0, 1, 4, 12, 24, 72, float('inf')]
labels = ['<1h', '1-4h', '4-12h', '12-24h', '1-3d', '3d+']
ttp_clean['bucket'] = pd.cut(ttp_hours, bins=bins, labels=labels)
print(ttp_clean['bucket'].value_counts().sort_index())

time_to_purchase.to_csv('outputs/time_to_purchase.csv', index=False)
ttp_clean.to_csv('outputs/time_to_purchase_clean.csv', index=False)

--- Time-to-Purchase (converted users) ---
Count:    1,543,759
Mean:     717.30 hrs
Median:   80.02 hrs
P25:      0.13 hrs
P75:      890.89 hrs
P95:      3720.83 hrs

Bucket distribution:
bucket
<1h       560339
1-4h       40449
4-12h      28688
12-24h     42768
1-3d       88209
3d+       783247
Name: count, dtype: int64


## 6. User-Level Features (Time-Aware, No Leakage)

Temporal split at the 80th time percentile. Features come from the past; labels from the future.

In [7]:
cutoff_time = con.execute("""
SELECT quantile_cont(event_time, 0.8) FROM events
""").fetchone()[0]

print(f'Prediction cutoff: {cutoff_time}')

con.execute(f"""
CREATE OR REPLACE TABLE user_features AS
SELECT
    user_id,
    COUNT(*)                                                  AS total_events,
    SUM(CASE WHEN event_type = 'view'     THEN 1 ELSE 0 END) AS views,
    SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) AS carts,
    SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS past_purchases,
    COUNT(DISTINCT product_id)                                AS unique_products,
    MAX(event_time) - MIN(event_time)                         AS active_span
FROM events
WHERE event_time <= TIMESTAMP '{cutoff_time}'
GROUP BY user_id
""")

labels_df = con.execute(f"""
SELECT
    user_id,
    CASE WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
         THEN 1 ELSE 0 END AS label
FROM events
WHERE event_time > TIMESTAMP '{cutoff_time}'
GROUP BY user_id
""").df()

features   = con.execute('SELECT * FROM user_features').df()
ml_dataset = features.merge(labels_df, on='user_id', how='left')
ml_dataset['label'] = ml_dataset['label'].fillna(0).astype(int)

print(f'\nDataset shape: {ml_dataset.shape}')
print(f'Purchase label balance: {ml_dataset["label"].value_counts().to_dict()}')

Prediction cutoff: 2020-04-08 18:34:16-04:00

Dataset shape: (10064887, 8)
Purchase label balance: {0: 9820624, 1: 244263}


## 7. Feature Engineering

In [8]:
ml_dataset['active_span_sec'] = (
    pd.to_timedelta(ml_dataset['active_span'], errors='coerce')
    .dt.total_seconds()
    .fillna(0)
)
ml_dataset.drop(columns=['active_span'], inplace=True)

# Derived features
ml_dataset['events_per_day']     = ml_dataset['total_events']   / (ml_dataset['active_span_sec'] / 86400 + 1)
ml_dataset['cart_to_view_ratio'] = ml_dataset['carts']          / (ml_dataset['views'] + 1)
ml_dataset['product_diversity']  = ml_dataset['unique_products'] / (ml_dataset['views'] + 1)
ml_dataset['has_past_purchase']  = (ml_dataset['past_purchases'] > 0).astype(int)
ml_dataset['purchase_rate']      = ml_dataset['past_purchases']  / (ml_dataset['total_events'] + 1)  # NEW

ml_dataset = ml_dataset.replace([np.inf, -np.inf], 0).fillna(0)

print('Features engineered:')
print(ml_dataset.describe().round(3))

ml_dataset.to_csv('outputs/ml_dataset.csv', index=False)

Features engineered:
            user_id  total_events         views         carts  past_purchases  \
count  1.006489e+07  1.006489e+07  1.006489e+07  1.006489e+07    1.006489e+07   
mean   5.681263e+08  2.145100e+01  2.019300e+01  9.210000e-01    3.370000e-01   
std    3.546363e+07  7.359600e+01  7.150100e+01  4.003000e+00    2.171000e+00   
min    1.030022e+07  1.000000e+00  0.000000e+00  0.000000e+00    0.000000e+00   
25%    5.421696e+08  2.000000e+00  2.000000e+00  0.000000e+00    0.000000e+00   
50%    5.677995e+08  5.000000e+00  4.000000e+00  0.000000e+00    0.000000e+00   
75%    5.958739e+08  1.700000e+01  1.600000e+01  0.000000e+00    0.000000e+00   
max    6.384533e+08  6.951500e+04  6.951500e+04  1.055000e+03    8.450000e+02   

       unique_products         label  active_span_sec  events_per_day  \
count     1.006489e+07  1.006489e+07     1.006489e+07    1.006489e+07   
mean      1.098800e+01  2.400000e-02     2.602792e+06    2.648000e+00   
std       3.068300e+01  1.5400

## 8. Purchase Prediction — Multi-Model Benchmark

Four classifiers are trained and evaluated:
- **LightGBM** and **XGBoost**: gradient boosting (top performers in tabular tasks)
- **Random Forest**: strong ensemble baseline
- **Logistic Regression**: linear baseline with StandardScaler

Primary metric: **AUC-ROC** (ranking). Secondary: **Average Precision** (imbalanced precision-recall).
All models use 5-fold stratified cross-validation.

In [13]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_auc_score,
    average_precision_score, confusion_matrix
)
import xgboost as xgb
import lightgbm as lgb

FEATURE_COLS = [
    'total_events', 'views', 'carts', 'unique_products',
    'active_span_sec', 'events_per_day',
    'cart_to_view_ratio', 'product_diversity',
    'has_past_purchase', 'purchase_rate'
]

X = ml_dataset[FEATURE_COLS].copy()
y = ml_dataset['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_leaf=5,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.05,
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100, max_depth=8, learning_rate=0.05,
        random_state=42, n_jobs=-1, verbose=-1
    ),
    'LogisticReg': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=500, random_state=42))
    ])
}

skf     = StratifiedKFold(3, shuffle=True, random_state=42)
results = {}

print('--- Model Comparison ---')
for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = model.predict(X_test)
    auc     = roc_auc_score(y_test, y_proba)
    ap      = average_precision_score(y_test, y_proba)
    cv      = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=1)
    report  = classification_report(y_test, y_pred, output_dict=True)
    results[name] = {
        'AUC-ROC':       round(auc, 4),
        'Avg-Precision': round(ap, 4),
        'CV_AUC_mean':   round(cv.mean(), 4),
        'CV_AUC_std':    round(cv.std(), 4),
        'F1_macro':      round(report['macro avg']['f1-score'], 4),
    }
    print(f'  {name:15s}  AUC={auc:.4f}  AP={ap:.4f}  CV={cv.mean():.4f}±{cv.std():.4f}')

results_df = pd.DataFrame(results).T
print('\n', results_df)
results_df.to_csv('outputs/model_comparison.csv')

--- Model Comparison ---
  RandomForest     AUC=0.7929  AP=0.1199  CV=0.7932±0.0007
  XGBoost          AUC=0.7954  AP=0.1212  CV=0.7959±0.0006
  LightGBM         AUC=0.7953  AP=0.1212  CV=0.7958±0.0006
  LogisticReg      AUC=0.7717  AP=0.1079  CV=0.7733±0.0005

               AUC-ROC  Avg-Precision  CV_AUC_mean  CV_AUC_std  F1_macro
RandomForest   0.7929         0.1199       0.7932      0.0007    0.4941
XGBoost        0.7954         0.1212       0.7959      0.0006    0.4940
LightGBM       0.7953         0.1212       0.7958      0.0006    0.4939
LogisticReg    0.7717         0.1079       0.7733      0.0005    0.4945


## 9. Feature Importance (Best Model + Random Forest)

In [14]:
# LightGBM importance
lgbm_fi = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': models['LightGBM'].feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('--- Feature Importance (LightGBM) ---')
print(lgbm_fi.to_string(index=False))
lgbm_fi.to_csv('outputs/feature_importance_lgbm.csv', index=False)

xgb_fi = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': models['XGBoost'].feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('--- Feature Importance (XGBoost) ---')
print(xgb_fi.to_string(index=False))
xgb_fi.to_csv('outputs/feature_importance_xgb.csv', index=False)

# Random Forest importance
rf_fi = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': models['RandomForest'].feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('\n--- Feature Importance (Random Forest) ---')
print(rf_fi.to_string(index=False))
rf_fi.to_csv('outputs/feature_importance_rf.csv', index=False)

--- Feature Importance (LightGBM) ---
           feature  importance
   active_span_sec        1201
 product_diversity         361
     purchase_rate         309
   unique_products         262
    events_per_day         260
cart_to_view_ratio         222
      total_events         154
             carts         146
             views          48
 has_past_purchase          37
--- Feature Importance (XGBoost) ---
           feature  importance
 has_past_purchase    0.355268
             carts    0.284165
      total_events    0.181314
   unique_products    0.075321
   active_span_sec    0.057947
     purchase_rate    0.022367
cart_to_view_ratio    0.012940
 product_diversity    0.006083
    events_per_day    0.002863
             views    0.001734

--- Feature Importance (Random Forest) ---
           feature  importance
   active_span_sec    0.380479
     purchase_rate    0.171336
 has_past_purchase    0.094434
cart_to_view_ratio    0.090962
             carts    0.068099
      total_e

## 10. SHAP Feature Attribution

SHAP (SHapley Additive exPlanations) values provide model-agnostic feature importance
that accounts for feature interactions and non-linearities, unlike raw split-based importance.

In [16]:
import shap

explainer   = shap.TreeExplainer(models['LightGBM'])
X_sample    = X_test.sample(min(2000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_sample)

# LightGBM binary: shap_values may be a list [neg_class, pos_class]
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap_importance = pd.DataFrame({
    'feature':       FEATURE_COLS,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('--- SHAP Feature Importance (LightGBM) ---')
print(shap_importance.to_string(index=False))
shap_importance.to_csv('outputs/shap_importance.csv', index=False)

--- SHAP Feature Importance (LightGBM) ---
           feature  mean_abs_shap
   unique_products       0.295413
   active_span_sec       0.281700
      total_events       0.257100
             carts       0.233952
 product_diversity       0.091867
     purchase_rate       0.062811
cart_to_view_ratio       0.049921
 has_past_purchase       0.047935
    events_per_day       0.036190
             views       0.008707


## 11. Purchase Probabilities (Best Model: XGBoost)

In [19]:
# Use best model (LightGBM) for scoring
best_model = models['XGBoost']
ml_dataset['purchase_probability'] = best_model.predict_proba(X)[:, 1]

print('--- Purchase Probability Distribution ---')
bins_p   = [0, .2, .4, .6, .8, 1.0]
labels_p = ['0-20%', '20-40%', '40-60%', '60-80%', '80-100%']
print(
    pd.cut(ml_dataset['purchase_probability'], bins=bins_p, labels=labels_p)
    .value_counts()
    .sort_index()
)

ml_dataset[['user_id', 'purchase_probability']].to_csv(
    'outputs/purchase_probabilities.csv', index=False
)
print('\nPurchase probabilities saved')

--- Purchase Probability Distribution ---
purchase_probability
0-20%      9993636
20-40%       70158
40-60%        1093
60-80%           0
80-100%          0
Name: count, dtype: int64

Purchase probabilities saved


## 12. Cart Abandonment Model

In [20]:
abandonment = con.execute("""
SELECT
    user_id,
    CASE
        WHEN SUM(CASE WHEN event_type = 'cart'     THEN 1 ELSE 0 END) > 0
         AND SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) = 0
        THEN 1 ELSE 0
    END AS abandoned
FROM events
GROUP BY user_id
""").df()

ab_dataset           = ml_dataset.merge(abandonment, on='user_id', how='left')
ab_dataset['abandoned'] = ab_dataset['abandoned'].fillna(0).astype(int)

carted_users = ab_dataset[ab_dataset['carts'] > 0].copy()
print(f'Carted users: {len(carted_users):,}  |  Abandoned: {carted_users["abandoned"].sum():,}')

X_ab = carted_users[FEATURE_COLS].copy()
y_ab = carted_users['abandoned']

X_ab_train, X_ab_test, y_ab_train, y_ab_test = train_test_split(
    X_ab, y_ab, test_size=0.2, random_state=42, stratify=y_ab
)

abandon_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=8, learning_rate=0.05,
    random_state=42, n_jobs=-1, verbose=-1
)
abandon_model.fit(X_ab_train, y_ab_train)

y_ab_proba = abandon_model.predict_proba(X_ab_test)[:, 1]
y_ab_pred  = abandon_model.predict(X_ab_test)

ab_auc = roc_auc_score(y_ab_test, y_ab_proba)
ab_ap  = average_precision_score(y_ab_test, y_ab_proba)

print(f'\n--- Cart Abandonment Model (LightGBM) ---')
print(f'AUC-ROC: {ab_auc:.4f}  |  Average Precision: {ab_ap:.4f}')
print()
print(classification_report(y_ab_test, y_ab_pred, digits=4))

carted_users = carted_users.copy()
carted_users['abandon_probability'] = abandon_model.predict_proba(X_ab)[:, 1]
carted_users[['user_id', 'abandon_probability']].to_csv(
    'outputs/abandon_probabilities.csv', index=False
)
print('Abandonment probabilities saved')

Carted users: 2,150,993  |  Abandoned: 958,531

--- Cart Abandonment Model (LightGBM) ---
AUC-ROC: 0.9893  |  Average Precision: 0.9785

              precision    recall  f1-score   support

           0     1.0000    0.9654    0.9824    238493
           1     0.9587    1.0000    0.9789    191706

    accuracy                         0.9808    430199
   macro avg     0.9794    0.9827    0.9807    430199
weighted avg     0.9816    0.9808    0.9808    430199

Abandonment probabilities saved


## 13. CLV Scoring

In [22]:
ml_dataset['clv_score'] = (
    ml_dataset['purchase_probability']
    * np.log1p(ml_dataset['total_events'])
    * (ml_dataset['carts'] + 1)
    * (1 + ml_dataset['has_past_purchase'])
)

clv_min = ml_dataset['clv_score'].min()
clv_max = ml_dataset['clv_score'].max()
ml_dataset['clv_score_norm'] = (
    (ml_dataset['clv_score'] - clv_min) / (clv_max - clv_min + 1e-9) * 100
).round(2)

ml_dataset['clv_decile'] = pd.qcut(
    ml_dataset['clv_score_norm'].rank(method='first'),
    q=10,
    labels=[f'D{i}' for i in range(1, 11)]
)

print('--- CLV Score by Decile ---')
decile_summary = (
    ml_dataset.groupby('clv_decile', observed=True)['clv_score_norm']
    .agg(['mean', 'min', 'max', 'count'])
    .round(2)
)
print(decile_summary)
decile_summary.to_csv('outputs/clv_decile_summary.csv')

ml_dataset[['user_id', 'clv_score_norm', 'clv_decile']].to_csv(
    'outputs/clv_scores.csv', index=False
)
print('\nCLV scores saved')

--- CLV Score by Decile ---
            mean   min     max    count
clv_decile                             
D1          0.00  0.00    0.00  1006489
D2          0.00  0.00    0.00  1006489
D3          0.00  0.00    0.00  1006488
D4          0.00  0.00    0.00  1006489
D5          0.00  0.00    0.00  1006489
D6          0.00  0.00    0.00  1006488
D7          0.00  0.00    0.00  1006489
D8          0.00  0.00    0.00  1006488
D9          0.00  0.00    0.01  1006489
D10         0.08  0.01  100.00  1006489

CLV scores saved


In [23]:
import numpy as np
import pandas as pd

# 1. Ensure purchase_probability is continuous (NOT binary 0/1)
# If you haven't assigned it yet, use this (replace 'LightGBM' with your best model):
X_full = ml_dataset[FEATURE_COLS]
ml_dataset['purchase_probability'] = models['XGBoost'].predict_proba(X_full)[:, 1]

# 2. Calculate the raw CLV score
ml_dataset['clv_raw'] = (
    ml_dataset['purchase_probability']
    * np.log1p(ml_dataset['total_events'])
    * (ml_dataset['carts'] + 1)
    * (1 + ml_dataset['has_past_purchase'])
)

# 3. USE LOG-TRANSFORM to make the distribution more readable
# This handles the extreme skewness so D1-D8 aren't all 0.00
ml_dataset['clv_log'] = np.log1p(ml_dataset['clv_raw'])

clv_min = ml_dataset['clv_log'].min()
clv_max = ml_dataset['clv_log'].max()

ml_dataset['clv_score_norm'] = (
    (ml_dataset['clv_log'] - clv_min) / (clv_max - clv_min + 1e-9) * 100
).round(2)

# 4. Bin into deciles using rank to handle ties
ml_dataset['clv_decile'] = pd.qcut(
    ml_dataset['clv_score_norm'].rank(method='first'), 
    q=10,
    labels=[f'D{i}' for i in range(1, 11)]
)

print('--- Optimized CLV Score by Decile (Log-Normalized) ---')
decile_summary = (
    ml_dataset.groupby('clv_decile', observed=True)['clv_score_norm']
    .agg(['mean', 'min', 'max', 'count'])
    .round(2)
)
print(decile_summary)

# 5. Save results
decile_summary.to_csv('outputs/clv_decile_summary.csv')
ml_dataset[['user_id', 'clv_score_norm', 'clv_decile']].to_csv(
    'outputs/clv_scores.csv', index=False
)

--- Optimized CLV Score by Decile (Log-Normalized) ---
             mean   min     max    count
clv_decile                              
D1           0.01  0.00    0.01  1006489
D2           0.01  0.01    0.01  1006489
D3           0.01  0.01    0.04  1006488
D4           0.08  0.04    0.11  1006489
D5           0.16  0.11    0.21  1006489
D6           0.28  0.21    0.38  1006488
D7           0.53  0.38    0.72  1006489
D8           1.11  0.72    1.72  1006488
D9           3.65  1.72    7.04  1006489
D10         17.79  7.04  100.00  1006489


## 14. User Segmentation

In [24]:
def segment_user(row):
    if row['has_past_purchase'] and row['purchase_probability'] >= 0.6:
        return 'High-value buyer'
    elif row['has_past_purchase']:
        return 'Occasional buyer'
    elif row['carts'] > 0 and row['label'] == 0:
        return 'Cart abandoner'
    elif row['views'] > 0:
        return 'Browser'
    else:
        return 'Dormant'

ml_dataset['segment'] = ml_dataset.apply(segment_user, axis=1)

segment_summary = (
    ml_dataset.groupby('segment')
    .agg(
        users          = ('user_id',             'count'),
        avg_clv        = ('clv_score_norm',       'mean'),
        avg_purchase_p = ('purchase_probability', 'mean'),
        avg_carts      = ('carts',                'mean'),
        avg_views      = ('views',                'mean')
    )
    .round(3)
    .sort_values('avg_clv', ascending=False)
)

print('--- User Segments ---')
print(segment_summary)

segment_summary.to_csv('outputs/segment_summary.csv')
ml_dataset[['user_id', 'segment', 'purchase_probability', 'clv_score_norm']].to_csv(
    'outputs/user_segments.csv', index=False
)
print('\nSegments saved')

--- User Segments ---
                    users  avg_clv  avg_purchase_p  avg_carts  avg_views
segment                                                                 
Occasional buyer  1255029   13.987           0.082      5.468     66.469
Cart abandoner     958531    3.856           0.040      2.397     43.954
Dormant                90    1.082           0.038      1.767      0.000
Browser           7851237    0.323           0.013      0.014      9.895

Segments saved


## 15. Summary & Targeting Recommendations

In [25]:
print('=' * 55)
print('ANALYSIS COMPLETE — OUTPUT FILES')
print('=' * 55)

output_files = [
    ('event_breakdown.csv',             'Event type frequency table'),
    ('transition_probs.csv',            'Markov transition probabilities'),
    ('transition_matrix.csv',           'Pivoted transition matrix'),
    ('top_transitions.csv',             'Top 10 sequential transitions'),
    ('funnel.csv',                      'Conversion funnel stage counts'),
    ('time_to_purchase.csv',            'Raw time-to-purchase per user'),
    ('time_to_purchase_clean.csv',      'Converted users only'),
    ('ml_dataset.csv',                  'Full feature + label dataset'),
    ('model_comparison.csv',            'Cross-validated metrics all models'),
    ('feature_importance_lgbm.csv',     'LightGBM split-based importance'),
    ('feature_importance_rf.csv',       'Random Forest importance'),
    ('shap_importance.csv',             'SHAP mean absolute values'),
    ('purchase_probabilities.csv',      'Per-user purchase probability'),
    ('abandon_probabilities.csv',       'Per-user abandonment probability'),
    ('clv_scores.csv',                  'Per-user CLV score & decile'),
    ('clv_decile_summary.csv',          'CLV decile aggregate stats'),
    ('segment_summary.csv',             'Aggregate segment statistics'),
    ('user_segments.csv',               'Segment assignment + scores'),
]

for fname, desc in output_files:
    path   = f'outputs/{fname}'
    exists = os.path.exists(path)
    print(f'  {"OK" if exists else "MISSING":6}  {fname:42}  {desc}')

print()
print('TARGETING RECOMMENDATIONS')
print('-' * 55)

recs = [
    ('Cart abandoners',
     'Re-engage within 4h. Target abandon_prob > 0.6 with free-shipping incentive.'),
    ('Browsers',
     'Prioritise cart_to_view_ratio > 0.12 and unique_products > 3 with urgency messaging.'),
    ('Occasional buyers',
     'Cross-sell via post-purchase → view path (~97% transition). Recommend related categories.'),
    ('High-value buyers',
     'Retention & loyalty. Personalise using browsed product_ids from event history.'),
    ('Dormant',
     'Win-back only for clv_score_norm > 40. Suppress others to reduce CAC.'),
]
for seg, rec in recs:
    print(f'\n  [{seg}]')
    print(f'  {rec}')

ANALYSIS COMPLETE — OUTPUT FILES
  OK      event_breakdown.csv                         Event type frequency table
  OK      transition_probs.csv                        Markov transition probabilities
  OK      transition_matrix.csv                       Pivoted transition matrix
  OK      top_transitions.csv                         Top 10 sequential transitions
  OK      funnel.csv                                  Conversion funnel stage counts
  OK      time_to_purchase.csv                        Raw time-to-purchase per user
  OK      time_to_purchase_clean.csv                  Converted users only
  OK      ml_dataset.csv                              Full feature + label dataset
  OK      model_comparison.csv                        Cross-validated metrics all models
  OK      feature_importance_lgbm.csv                 LightGBM split-based importance
  OK      feature_importance_rf.csv                   Random Forest importance
  OK      shap_importance.csv                         S